<center><h1><strong><font color="blue"> Data Engineering for Data Science - Week 07</font></strong></h1></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Normalization and Case Study</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

**Topics**:

* Normal Forms: 1NF, 2NF, 3NF, & BCNF
* Denormalization: When and why to break the rules.
* **case Study**:
   - Normalizations
   - Basic Query
   - Join
   - Aggregations

<center><h2><strong><font color="blue">Prologue/Motivation ~ Normalization</font></strong></h2></center>

<center><img alt="" src="images/deds-05/motivation-normalization.png" style="height: 600px;"/></center>

> *An **orphan record** problem in an unnormalized database occurs when data that should be logically dependent on another entity becomes disconnected due to the absence of enforced relationships or constraints. For example, a transaction row may reference a customer or product that no longer exists (or was inconsistently modified), because all data resides in a single flat structure without referential integrity. This leads to inconsistencies, invalid references, and unreliable query results, especially during updates or deletions where related data is not systematically maintained*.

<center><h2><strong><font color="blue">Case Study: Global E-Commerce Mini-Store</font></strong></h2></center>

## 1. Overview
This case study examines a simplified **Global Mini-Store**, an online platform where customers from multiple countries purchase technology gadgets. The system currently relies on a **single flat-file structure** (spreadsheet-like table) to record all transactions. While this approach is straightforward and easy to implement at an early stage, it introduces **serious data integrity and scalability issues** as the system grows.

## 2. Problem Context
- Each row represents a **single line item** within a customer order.
- All relevant information—order, customer, product, and transaction—is stored in **one table**.

<center><img alt="" src="images/deds-06/mini-store-case-study.png" style="height: 480px;"/></center>

### Data Components Stored in a Single Table
1. **Order Details**: `order_id`, `order_date`
2. **Customer Information**: `customer_name`, `customer_email`, `customer_country`
3. **Product Details**: `product_id`, `product_name`, `product_category`
4. **Transaction Information**: `unit_price`, `quantity`

<center><h2><strong><font color="blue"> Export-Import Data</font></strong></h2></center>

<center><h3><strong><font color="red"> Please clone (“pull”) the course GitHub repository and locate the file</font><font color="green"> "unnormalized-store-table-example.sql" </font><font color="red"> within the "data" folder</font></strong></h3></center>

### import the SQL sample data to your MySQL server using phpMyadmin as in the previous session.

<center><img alt="" src="images/deds-06/miniStore-initial-table.jpg" style="height: 600px;"/></center>

<center><h3><strong><font color="red"> Pause and carefully examine the table. What are the issues exist in this unstructured (monolithic) table design?</font></strong></h3></center>

<center><img alt="" src="images/stop-exercise-learn.jpg" style="height: 320px;"/></center>

> Hints: *Find possible insert, delete, & update anomalies*

### Database Anomalies
Anomalies are problems that arise during the manipulation of a poorly structured table. They are generally categorized into three types:

| Anomaly Type | Description | Research Example |
| :--- | :--- | :--- |
| **Update Anomaly** | Occurs when a change to a single data value requires multiple rows to be updated. | If a Lab Location changes, and it is stored in every row of an "Experiments" table, failing to update every row leads to inconsistent records. |
| **Insertion Anomaly** | Occurs when certain data cannot be recorded unless other, unrelated data is also provided. | You cannot record a new Researcher's profile until they are assigned to an active Experiment, because the primary key requires an Experiment ID. |
| **Deletion Anomaly** | Occurs when deleting one piece of information unintentionally results in the loss of other unrelated data. | Deleting the last Experiment entry for a specific Lab might accidentally delete all contact information for that Lab. |

<center><h2><strong><font color="blue"> Phase 1: Achieving First Normal Form (1NF) </font></strong></h2></center>

## Conceptual Foundation
In the context of database normalization, **First Normal Form (1NF)** enforces *data atomicity* as a fundamental constraint. A relation is said to satisfy 1NF if it adheres to the following conditions:
1. **Atomic Attributes**  
   Each column must contain indivisible (atomic) values—no composite or multi-valued entries.
2. **No Repeating Groups**  
   Attributes must not store multiple values in a single cell (e.g., comma-separated lists).
3. **Row Uniqueness**  
   Every record must be uniquely identifiable via a **Primary Key**.

## Problem Identification
Consider the `online_retail_raw` table. Several records violate 1NF due to the presence of **multi-valued attributes**. For instance:
| order_id | product_id | product_name                  | unit_price     | quantity |
|----------|------------|-------------------------------|----------------|----------|
| 101      | 201, 205   | Wireless Mouse, Mechanical Keyboard | 25.00, 75.00 | 2, 1     |

### Observations
- Attributes such as `product_id`, `product_name`, `unit_price`, and `quantity` contain **comma-separated values**.
- This violates atomicity and introduces **repeating groups**.
- Consequently, standard SQL operations (e.g., filtering, aggregation, joins) become inefficient or infeasible.

## 1NF Normalization Procedure
### Step 1: Decompose Multi-Valued Attributes
Transform each multi-valued cell into multiple rows such that each row contains **exactly one value per attribute**.

#### Example Transformation (Order 101)
| order_id | product_id | product_name        | unit_price | quantity |
|----------|------------|---------------------|------------|----------|
| 101      | 201        | Wireless Mouse      | 25.00      | 2        |
| 101      | 205        | Mechanical Keyboard | 75.00      | 1        |

### Step 2: Enforce Uniform Row Structure
Apply the decomposition process across all records:
- Each product within an order becomes a separate row.
- Attributes such as `customer_name` and `customer_email` may repeat across rows—this is acceptable in 1NF.
- The resulting table represents **transaction-level granularity**.

### Step 3: Define the Primary Key
Post-transformation:
- `order_id` is **not unique** (one order → multiple products).
- `product_id` is **not unique** (one product → multiple orders).

#### Solution: Composite Primary Key
Define a **composite key**:
```sql
PRIMARY KEY (order_id, product_id)
````
This ensures that each row is uniquely identifiable.

## Result: 1NF-Compliant Table
* The original dataset (30 order records) is transformed into a larger set of **atomic transactional rows**.
* Each row now represents a single product within a specific order.
* The structure supports efficient querying, filtering, and aggregation.

<center><img alt="" src="images/deds-06/miniStore-initial-table.jpg" style="height: 600px;"/></center>

<center><h2><strong><font color="green">Start designing database schema for your thesis project</font></strong></h2></center>

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

## Next Week's Discussions

* Entity-Relationship Diagrams (ERD).
* Data Acquisition II – Web Scraping & Crawling
   - Legal & Ethics of Scraping (robots.txt, Terms of Service).
   - Data Streaming
   - Normalization Assignment

<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-database-primary-keys.jpeg" style="height: 600px;"/></center>